In [ ]:
from math import pi
import numpy as np
import scipy as sp
#from qiskit.opflow import CircuitSampler, StateFn, AerPauliExpectation, PauliSumOp
from qiskit.quantum_info import Pauli
from qiskit.circuit.library import PauliEvolutionGate
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit import Parameter
from qiskit_algorithms.optimizers import COBYLA, SLSQP, SPSA, L_BFGS_B
from qiskit.circuit.library import TwoLocal
from qiskit_algorithms.optimizers import SciPyOptimizer
from qiskit.primitives import Sampler # new
#from qiskit.opflow.primitive_ops import PauliSumOp old
from qiskit.quantum_info import SparsePauliOp,PauliList
from qiskit.circuit import QuantumCircuit, QuantumRegister
#from qiskit.opflow.expectations import ExpectationFactory
from qiskit.primitives import Estimator # new
#from qiskit.opflow.state_fns import CircuitStateFn
#from qiskit.quantum_info import States old
import time
import pandas as pd
from qiskit.providers.basic_provider import BasicProvider #new
from multiprocessing import Pool
import multiprocessing as mp
from qiskit_algorithms.optimizers import SciPyOptimizer
import scipy as sp
from scipy.optimize import differential_evolution, Bounds
import os
from scipy.optimize import minimize

In [ ]:
def create_ham_str(nqubits):

    # Create a list of terms for the hamiltonian (open boundary conditions)
    # Inputs: nqubits (int), number of qubits
    # Outputs: ham (list), hamiltonian 

    ham = []

    for i in range(nqubits-1):

        term = ''

        for j in range(nqubits-i-2):

            term += 'I'

        for j in range(nqubits-i-2,nqubits-i):

            term += 'Y'  # Choose a Pauli matrix, i.e., X, Y or Z

        for j in range(nqubits-i,nqubits):

            term += 'I'

        ham.append(term)

    return ham



def evaluate_expectation(parameters_values):
    # Function to evaluate the expectation value of the Hamiltonian for a given set of parameters
    # Inputs: parameter_values (ndarray), parameter values
    # Outputs: result (float), energy value

    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars =  list(value_dict.values())
    expectation_value = estimator.run(ansatz,qubit_op,pars).result().values
    return np.real(expectation_value)

In [ ]:
import numpy as np
import time
from scipy.optimize import minimize
k=8
def evaluate(parameters_values):
    # Function to evaluate the expectation value of the Hamiltonian for a given set of parameters
    # Inputs: parameter_values (ndarray), parameter values
    # Outputs: result (float), energy value

    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars =  list(value_dict.values())
    expectation_value = estimator.run(ansatz,qubit_op,pars).result().values
    return np.real(expectation_value)

# Define your initial population generation function
def generate_population(pop_size, dimension, var_min, var_max):
    population = []
    for _ in range(pop_size):
        individual = np.random.uniform(var_min, var_max, dimension)
        fitness = evaluate(individual)
        population.append(Individual(individual, fitness))
    return population

# Define your individual class
class Individual:
    def __init__(self, params, fitness):
        self.params = params
        self.fitness = fitness

# Define the function to evaluate expectation for COBYLA
def evaluate_expectation(x):
    return evaluate(x)

#################################################### Hamiltonian ###########################################################
    
qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
qubits=k
depth=1
#SOMA parameters
prt = 0.2
path_lenght = 2
step = 0.11
migrations = 20
pop_size = 15

#general parameters
dimension = qubits*4
min_s = [-3.14]*dimension
max_s = [3.14]*dimension
    
###################################################### Ansatz ##############################################################

q_init=QuantumCircuit(qubit_op.num_qubits)
for i in range(0,qubit_op.num_qubits):
        q_init.ry(np.pi/4,i)
        
ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
ansatz.compose(q_init,front=True,inplace=True)
#print(ansatz.decompose().draw(fold=-1))

print('ansatz_num_parameters=',ansatz.num_parameters)

##################################### Instructions for the energy evaluation ###############################################
from qiskit_aer.primitives import Estimator

# Simulations are noiseless and without sampling. 
#backend = Aer.get_backend('aer_simulator') old
device = BasicProvider().get_backend('basic_simulator')
coupling_map = device.configuration().coupling_map

# If a noise model is provided, the Aer primitives
# perform a "qasm" simulation
estimator = Estimator(
            run_options={"shots": 5120},
        )

starttime = time.time()  # Start the timer
print('Hello! iSOMA is working, please wait... ')
dimension = 4 * k  # Number of dimensions of the problem
# -------------- Control Parameters of SOMA -------------------------------
N_jump, Step = 10, 0.3  # Assign values ​​to variables: Step, PRT, PathLength
PopSize, Max_Migration, Max_FEs = 100, 100, 10 * dimension**4  # Assign values ​​to variables: PopSize, Max_Migration
m, n, s = 10, 5, 15
local_better, local_count = 0, 0
# -------------- The domain (search space) --------------------------------
VarMin, VarMax = -np.pi, np.pi  # Define the search range

population = generate_population(PopSize, dimension, VarMin, VarMax)
pop = np.array([ind.params for ind in population])
fitness = np.array([ind.fitness for ind in population])
FEs = PopSize  # Count the number of function evaluations
the_best_cost = np.min(fitness)  # Find the Global minimum fitness value
the_best_value = pop[np.argmin(fitness)]  # Find the Global minimum position
# ---------------- SOMA MIGRATIONS ----------------------------------------
best_cost_old = the_best_cost
Migration, Count = 0, 0  # Assign values ​​to variables: Migration
local_search_interval = 25  # Interval to perform local search
top_n_individuals = 3  # Number of top individuals to perform local search on

while np.abs(the_best_cost + qubits - 1) >= 1e-1:  # Terminate when reaching Max_Migration / User can change to Max_FEs
    Migration += 1  # Increase Migration value
    # ------------ Migrant selection: m -----------------------------------
    M = np.random.choice(range(PopSize), m, replace=False)  # Migrant selection: m
    M_sort = np.argsort(fitness[M])
    for j in range(n):  # Choose n individuals move toward the Leader
        Migrant = pop[M[M_sort[j]]].reshape(dimension, 1)  # Get the Migrant position (solution values) in the current population
        # ------------ Leader selection: k --------------------------------
        K = np.random.choice(range(PopSize), s, replace=False)  # Leader selection: k
        K_sort = np.argsort(fitness[K])
        Leader = pop[K[K_sort[1]]].reshape(dimension, 1)  # Get the Leader position (solution values) in the current population
        if M[M_sort[j]] == K[K_sort[1]]:  # Don't move if it is itself
            Leader = pop[K[K_sort[2]]].reshape(dimension, 1)  # Get the Leader position (solution values) in the current population
        # ------ Migrant move to Leader: Jumping --------------------------
        flag, move = 0, 1
        while (flag == 0) and (move <= N_jump):
            nstep = (N_jump - move + 1) * Step
            # ------ Update Control parameters: PRT -----------------------
            PRT = 0.1 + 0.9 * (FEs / Max_FEs)  # Update PRT parameter
            # ----- SOMA Mutation -----------------------------------------
            PRTVector = (np.random.rand(dimension, 1) < PRT) * 1  # If rand() < PRT, PRTVector = 1, else, 0
            offspring = Migrant + (Leader - Migrant) * nstep * PRTVector  # Jumping towards the Leader
            # ------------ Check and put individuals inside the search range if it's outside
            for rw in range(dimension):  # From row: Check
                if offspring[rw] < VarMin or offspring[rw] > VarMax:  # if outside the search range
                    offspring[rw] = VarMin + np.random.rand() * (VarMax - VarMin)  # Randomly put it inside
            # ------------ Evaluate the offspring and Update --------------
            new_cost = evaluate(offspring.flatten())  # Evaluate the offspring
            FEs += 1  # Count the number of function evaluations
            # ----- SOMA Accepting: Place the Best Offspring to Pop -------
            if new_cost <= fitness[M[M_sort[j]]]:  # Compare min_new_cost with fitness value of the moving individual
                flag = 1
                fitness[M[M_sort[j]]] = new_cost  # Replace the moving individual fitness value
                pop[M[M_sort[j]]] = offspring.flatten()  # Replace the moving individual position (solution values)
                if new_cost <= the_best_cost:  # Compare Current minimum fitness with Global minimum fitness
                    the_best_cost = new_cost  # Update Global minimum fitness value
                    the_best_value = offspring.flatten()  # Update Global minimum position
                else:
                    Count += 1
            move += 1
    if Count > PopSize * 50:
        if the_best_cost == best_cost_old:
            rat = round(0.1 * PopSize)
            pop_temp = VarMin + np.random.rand(dimension, rat) * (VarMax - VarMin)
            fit_temp = evaluate(pop_temp.flatten())
            FEs += rat
            D = np.random.choice(range(PopSize), rat, replace=False)
            pop[D] = pop_temp.T
            fitness[D] = fit_temp
        else:
            best_cost_old = the_best_cost
        Count = 0

    # Hybrid optimization: Perform local optimization every local_search_interval migrations
    if Migration % local_search_interval == 0:
        sorted_indices = np.argsort(fitness)[:top_n_individuals]  # Get the indices of the top N individuals
        for idx in sorted_indices:
            optimizer = SPSA(maxiter=300*k)
            optimizer2 = optimizer.minimize(fun=evaluate_expectation, x0= pop[idx].reshape((dimension,))) 
            local_opt_value = optimizer2.x
            local_opt_cost = optimizer2.fun
            local_count +=1

            # Compare the fitness of the local optimized solution with the original
            if local_opt_cost < fitness[idx]:
                fitness[idx] = local_opt_cost
                pop[idx] = local_opt_value
                # Update the global best if necessary
                if local_opt_cost < the_best_cost:
                    the_best_cost = local_opt_cost
                    the_best_value = local_opt_value
                    local_better +=1

# %%%%%%%%%%%%%%%%%%    E N D    S O M A     %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
endtime = time.time()  # Stop the timer
caltime = endtime - starttime  # Calculate the processing time
# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
# Show the information to User
print('Stop at Migration :  ', Migration)
print('The number of FEs :  ', FEs)
print('Processing time   :  ', caltime, '(s)')
print('The best cost     :  ', the_best_cost)
print('Solution values   :  ', the_best_value)
print('Local count:   ', local_count)
print('Local was better:   ', local_better)
timer = []
timer.append(caltime)


In [ ]:
import numpy as np
import time
from scipy.optimize import minimize

def evaluate(parameters_values):
    # Function to evaluate the expectation value of the Hamiltonian for a given set of parameters
    # Inputs: parameter_values (ndarray), parameter values
    # Outputs: result (float), energy value

    value_dict = dict(zip(ansatz.parameters, parameters_values))
    pars = list(value_dict.values())
    expectation_value = estimator.run(ansatz, qubit_op, pars).result().values
    return np.real(expectation_value)

# Define your initial population generation function
def generate_population(pop_size, dimension, var_min, var_max):
    population = []
    for _ in range(pop_size):
        individual = np.random.uniform(var_min, var_max, dimension)
        fitness = evaluate(individual)
        population.append(Individual(individual, fitness))
    return population

# Define your individual class
class Individual:
    def __init__(self, params, fitness):
        self.params = params
        self.fitness = fitness

# Define the function to evaluate expectation for COBYLA
def evaluate_expectation(x):
    return evaluate(x)

# Loop over different values of k (number of qubits)
for k in range(3, 11):
    dimension = k*4
    total_time = 0
    print(f"Running optimization for k = {k} (number of qubits)")

    #################################################### Hamiltonian ###########################################################
    
    qubit_op=SparsePauliOp(PauliList(create_ham_str(k)),coeffs=-1.0)
    qubits=k
    depth=1
    #SOMA parameters
    prt = 0.2
    path_lenght = 2
    step = 0.11
    migrations = 20
    pop_size = 15

    #general parameters
    dimension = qubits*4
    min_s = [-3.14]*dimension
    max_s = [3.14]*dimension
    
    ###################################################### Ansatz ##############################################################
    
    q_init=QuantumCircuit(qubit_op.num_qubits)
    for i in range(0,qubit_op.num_qubits):
            q_init.ry(np.pi/4,i)
            
    ansatz=TwoLocal(qubit_op.num_qubits, ['ry','rz'], 'cz', 'linear', reps=1, insert_barriers=True)
    ansatz.compose(q_init,front=True,inplace=True)
    #print(ansatz.decompose().draw(fold=-1))
    
    print('ansatz_num_parameters=',ansatz.num_parameters)
    
    ##################################### Instructions for the energy evaluation ###############################################
    from qiskit_aer.primitives import Estimator

    # Simulations are noiseless and without sampling. 
    #backend = Aer.get_backend('aer_simulator') old
    device = BasicProvider().get_backend('basic_simulator')
    coupling_map = device.configuration().coupling_map
 
    # If a noise model is provided, the Aer primitives
    # perform a "qasm" simulation
    estimator = Estimator(
                run_options={"shots": 5120},
            )

    for run in range(5):
        starttime = time.time()  # Start the timer
        print(f"Run {run + 1} for k = {k}")
        dimension = 4 * k  # Number of dimensions of the problem
        # -------------- Control Parameters of SOMA -------------------------------
        N_jump, Step = 10, 0.3  # Assign values ​​to variables: Step, PRT, PathLength
        PopSize, Max_Migration, Max_FEs = 100, 100, 10 * dimension**4  # Assign values ​​to variables: PopSize, Max_Migration
        m, n, s = 10, 5, 15
        local_better, local_count = 1, 1
        # -------------- The domain (search space) --------------------------------
        VarMin, VarMax = -np.pi, np.pi  # Define the search range

        population = generate_population(PopSize, dimension, VarMin, VarMax)
        pop = np.array([ind.params for ind in population])
        fitness = np.array([ind.fitness for ind in population])
        FEs = PopSize  # Count the number of function evaluations
        the_best_cost = np.min(fitness)  # Find the Global minimum fitness value
        the_best_value = pop[np.argmin(fitness)]  # Find the Global minimum position
        # ---------------- SOMA MIGRATIONS ----------------------------------------
        best_cost_old = the_best_cost
        Migration, Count = 0, 0  # Assign values ​​to variables: Migration
        local_search_interval = 2*  k**2  # Adaptive local search interval
        top_n_individuals = 3  # Number of top individuals to perform local search on

        while np.abs(the_best_cost + qubits - 1) >= 1e-1:  # Terminate when reaching Max_Migration / User can change to Max_FEs
            Migration += 1  # Increase Migration value
            # ------------ Migrant selection: m -----------------------------------
            M = np.random.choice(range(PopSize), m, replace=False)  # Migrant selection: m
            M_sort = np.argsort(fitness[M])
            for j in range(n):  # Choose n individuals move toward the Leader
                Migrant = pop[M[M_sort[j]]].reshape(dimension, 1)  # Get the Migrant position (solution values) in the current population
                # ------------ Leader selection: k --------------------------------
                K = np.random.choice(range(PopSize), s, replace=False)  # Leader selection: k
                K_sort = np.argsort(fitness[K])
                Leader = pop[K[K_sort[1]]].reshape(dimension, 1)  # Get the Leader position (solution values) in the current population
                if M[M_sort[j]] == K[K_sort[1]]:  # Don't move if it is itself
                    Leader = pop[K[K_sort[2]]].reshape(dimension, 1)  # Get the Leader position (solution values) in the current population
                # ------ Migrant move to Leader: Jumping --------------------------
                flag, move = 0, 1
                while (flag == 0) and (move <= N_jump):
                    nstep = (N_jump - move + 1) * Step
                    # ------ Update Control parameters: PRT -----------------------
                    PRT = 0.1 + 0.9 * (FEs / Max_FEs)  # Update PRT parameter
                    # ----- SOMA Mutation -----------------------------------------
                    PRTVector = (np.random.rand(dimension, 1) < PRT) * 1  # If rand() < PRT, PRTVector = 1, else, 0
                    offspring = Migrant + (Leader - Migrant) * nstep * PRTVector  # Jumping towards the Leader
                    # ------------ Check and put individuals inside the search range if it's outside
                    for rw in range(dimension):  # From row: Check
                        if offspring[rw] < VarMin or offspring[rw] > VarMax:  # if outside the search range
                            offspring[rw] = VarMin + np.random.rand() * (VarMax - VarMin)  # Randomly put it inside
                    # ------------ Evaluate the offspring and Update --------------
                    new_cost = evaluate(offspring.flatten())  # Evaluate the offspring
                    FEs += 1  # Count the number of function evaluations
                    # ----- SOMA Accepting: Place the Best Offspring to Pop -------
                    if new_cost <= fitness[M[M_sort[j]]]:  # Compare min_new_cost with fitness value of the moving individual
                        flag = 1
                        fitness[M[M_sort[j]]] = new_cost  # Replace the moving individual fitness value
                        pop[M[M_sort[j]]] = offspring.flatten()  # Replace the moving individual position (solution values)
                        if new_cost <= the_best_cost:  # Compare Current minimum fitness with Global minimum fitness
                            the_best_cost = new_cost  # Update Global minimum fitness value
                            the_best_value = offspring.flatten()  # Update Global minimum position
                        else:
                            Count += 1
                    move += 1
            if Count > PopSize * 50:
                if the_best_cost == best_cost_old:
                    rat = round(0.1 * PopSize)
                    pop_temp = VarMin + np.random.rand(dimension, rat) * (VarMax - VarMin)
                    fit_temp = evaluate(pop_temp.flatten())
                    FEs += rat
                    D = np.random.choice(range(PopSize), rat, replace=False)
                    pop[D] = pop_temp.T
                    fitness[D] = fit_temp
                else:
                    best_cost_old = the_best_cost
                Count = 0

            # Adaptive hybrid optimization: Perform local optimization
            if Migration % local_search_interval == 0:
                sorted_indices = np.argsort(fitness)[:top_n_individuals]  # Get the indices of the top N individuals
                for idx in sorted_indices:
                    optimizer2 = minimize(evaluate_expectation, pop[idx].reshape((dimension,)), method='COBYLA', options={'maxiter': 100000})
                    local_opt_value = optimizer2.x
                    local_opt_cost = optimizer2.fun
                    local_count += 1

                    # Compare the fitness of the local optimized solution with the original
                    if local_opt_cost < fitness[idx]:
                        fitness[idx] = local_opt_cost
                        pop[idx] = local_opt_value
                        # Update the global best if necessary
                        if local_opt_cost < the_best_cost:
                            the_best_cost = local_opt_cost
                            the_best_value = local_opt_value
                            local_better += 1

        # %%%%%%%%%%%%%%%%%%    E N D    S O M A     %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
        endtime = time.time()  # Stop the timer
        caltime = endtime - starttime  # Calculate the processing time
        total_time += caltime
        # %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
        # Show the information to User
        print('Stop at Migration :  ', Migration)
        print('The number of FEs :  ', FEs)
        print('Processing time   :  ', caltime, '(s)')
        print('The best cost     :  ', the_best_cost)
        #print('Solution values   :  ', the_best_value)
        print('Local count       :  ', local_count)
        print('Local was better  :  ', local_better)

    average_time = total_time / 5
    print(f"Average processing time for k = {k} (number of qubits): {average_time:.2f} seconds\n")
